In [ ]:
"""
Arquitetura base - GPT-2

funções utils 
1 - get_lr
2 - save_model
3 - confg
4 - estimate_loss
"""

import math
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

class LayerNorm(nn.Module):
    """LayerNorm com suporte a bias opcional."""
    
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
        self.ndim = ndim  # Guarda a dimensão
        #self.eps = eps  # Guarda o valor de epsilon
    def forward(self, input):
        # CORREÇÃO: usar a dimensão correta (a última dimensão)
        return F.layer_norm(input, (self.ndim,), self.weight, self.bias, 1e-5)

class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = MultiHead(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class MultiHead(nn.Module):
    """Implementação MultiHead Attention para teste."""
    
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.num_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.num_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = torch.nn.functional.scaled_dot_product_attention(
                q, k, v, attn_mask=None, 
                dropout_p=self.dropout if self.training else 0, 
                is_causal=True
            )
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v
            
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class ModeloCompleto(nn.Module):
    """Versão simplificada do modelo para teste."""
    
    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        # Embeddings
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.wpe = nn.Embedding(config.block_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        
        # Blocks
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.num_layer)])
        
        # Final layer norm
        self.ln_f = LayerNorm(config.n_embd, bias=config.bias)
        
        # Language model head
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        
        # Weight tying
        self.wte.weight = self.lm_head.weight
        
        # Inicialização
        self.apply(self._init_weights)
        
        # Inicialização especial para projeções residuais
        for name, param in self.named_parameters():
            if name.endswith('c_proj.weight'):
                torch.nn.init.normal_(param, mean=0.0, std=0.02/math.sqrt(2 * config.num_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, idx, targets=None):
        b, t = idx.size()
        assert t <= self.config.block_size
        
        pos = torch.arange(0, t, dtype=torch.long, device=idx.device)
        
        tok_emb = self.wte(idx)
        pos_emb = self.wpe(pos)
        x = self.drop(tok_emb + pos_emb)
        
        for block in self.blocks:
            x = block(x)
        
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Take a conditioning sequence of indices idx (LongTensor of shape (b,t)) and complete
        the sequence max_new_tokens times, feeding the predictions back into the model each time.
        Most likely you'll want to make sure to be in model.eval() mode of operation for this.
        """
        for _ in range(max_new_tokens):
            # if the sequence context is growing too long we must crop it at block_size
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            # forward the model to get the logits for the index in the sequence
            logits, _ = self(idx_cond)
            # pluck the logits at the final step and scale by desired temperature
            logits = logits[:, -1, :] / temperature
            # optionally crop the logits to only the top k options
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            # apply softmax to convert logits to (normalized) probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # append sampled index to the running sequence and continue
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

# Configuração do modelo
class Config:
    def __init__(self, config_dict):
        self.vocab_size = config_dict.get("vocab_size", 10000)
        self.n_embd = config_dict.get("embedding_dim", 128)
        self.num_head = config_dict.get("num_heads", 2)
        self.num_layer = config_dict.get("num_layers", 2)
        self.dropout = config_dict.get("dropout", 0.0)
        self.bias = config_dict.get("bias", False)
        self.block_size = config_dict.get("block_size", config_dict.get("seq_len", 128))
        self.num_experts = config_dict.get("num_experts", 8)
        self.num_experts_per_tok = config_dict.get("num_experts_per_tok", 4)
        self.moe_aux_loss_coef = config_dict.get("moe_aux_loss_coef", 0.01)


# Save final
def save_model(model_save_path, model):
    """Salva o modelo em FP16"""
    Path(model_save_path).parent.mkdir(parents=True, exist_ok=True)
    state_dict = model.state_dict()
    fp16_state_dict = {}
    for key, value in state_dict.items():
        if value.is_floating_point():
            fp16_state_dict[key] = value.half()
        else:
            fp16_state_dict[key] = value
    
    torch.save(fp16_state_dict, model_save_path)
    print(f"Modelo salvo em FP16: {model_save_path}")

# Learning rate dinamico - boa pratica
def get_lr( step: int, lr: float, warmup_steps: int, max_steps: int, min_lr: float) -> float:
    """Warmup linear seguido de cosine decay até min_lr."""
    if warmup_steps > 0 and step < warmup_steps:
        return min(lr * (step + 1) / warmup_steps, lr)

    if step >= max_steps:
        return min_lr

    decay_ratio = (step - warmup_steps) / max(1, max_steps - warmup_steps)
    decay_ratio = min(max(decay_ratio, 0.0), 1.0)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    # Protege contra um erro de arredondamento de ponto flutuante na fronteira.
    return min(max(min_lr + coeff * (lr - min_lr), min_lr), lr)


@torch.no_grad()
def estimate_loss(model, loader, num_batches: int, device, amp_dtype, use_amp: bool) -> float:
    """Calcula a loss media em batches reservados para validacao."""
    was_training = model.training
    model.eval()
    losses = []
    for _ in range(num_batches):
        x, y = loader.get_batch()
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            _, loss = model(x, y)
        losses.append(loss.item())
    if was_training:
        model.train()
    return sum(losses) / len(losses)

# Save final
def save_model(model_save_path, model):
    """Salva o modelo em FP16"""
    path = Path(model_save_path)
    if str(path.parent) != ".":
        path.parent.mkdir(parents=True, exist_ok=True)
    
    state_dict = model.state_dict()
    fp16_state_dict = {}
    for key, value in state_dict.items():
        if value.is_floating_point():
            fp16_state_dict[key] = value.half()
        else:
            fp16_state_dict[key] = value
    
    torch.save(fp16_state_dict, path)
    print(f"Modelo salvo em FP16: {path}")

In [ ]:
#!/usr/bin/env python3
"""
Versão minimalista para baixar train.bin
"""

import os
from huggingface_hub import hf_hub_download

# Configure aqui
REPO_ID = "marcos-j-leemes/tinyS"
TOKEN = "///"  # Coloque seu token aqui | acelera o downloading
FILENAME = "train.bin"

# Baixa
print(f"Baixando {FILENAME}...")
path = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    repo_type="dataset",
    token=TOKEN,
    local_dir=".",
)

print(f" Baixado para: {path}")

In [ ]:
"""
Treinamento com múltiplas GPUs usando DistributedDataParallel (DDP).

Rodar com:
    torchrun --nproc_per_node=2 train_ddp.py

Principais mudanças em relação à versão com DataParallel:
  1. Multiprocessing real: um processo por GPU (torchrun cria e gerencia isso).
  2. init_process_group + destroy_process_group no início/fim.
  3. rank / local_rank / world_size lidos das variáveis de ambiente que o torchrun injeta.
  4. Cada processo usa sua própria GPU (device = cuda:{local_rank}), não mais um único
     device controlando todas.
  5. get_batch agora restringe o sorteio de índices à fatia lógica do dataset que
     pertence a este rank (equivalente manual a um DistributedSampler).
  6. Modelo empacotado com DDP(model, device_ids=[local_rank]) em vez de
     nn.DataParallel(model). O DDP sincroniza gradientes via all-reduce
     automaticamente durante o loss.backward().
  7. BATCH_SIZE agora é o batch LOCAL de cada GPU (não é mais fatiado como no
     DataParallel). O batch efetivo global passa a ser BATCH_SIZE * world_size.
  8. Prints, evaluate() e save_model() só rodam no rank 0, para não duplicar
     logs/checkpoints (cada processo faria isso senão).
  9. Removido o DebugModel/prints de shape por GPU — não é mais necessário, já
     que agora sabemos exatamente o que cada rank recebe (seu batch completo).
"""

# imports library
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import os
import numpy as np
import time

# Dados
DATASET = "" #os.path.dirname(os.path.abspath(__file__))
FILENAME = "train.bin"

# Configurações do modelo
VOCAB_SIZE = 10001
EMBEDDING_DIM = 128
NUM_HEADS = 2
NUM_LAYERS = 2
BLOCK_SIZE = 1040
DROPOUT = 0.0

# configurações do treinamento
MAX_STEPS = 250
BATCH_SIZE = 12          # <-- agora é o batch LOCAL por GPU (não fatiado)
GRAD_ACCUM_STEPS = 30
WEIGHT_DECAY = 0.0001
WARMUP_STEPS = 10
LEARNING_RATE = 1e-3
MIN_LR = 1e-5
GRAD_CLIP = 1.0

# logs
PRINT_INTERVAL = 10
EVAL_INTERVAL = 100

# Configuração do hardware
USE_AMP = True
USE_BF16 = False
TORCH_COMPILE = True

# save model
STEP_SAVE_INTERVAL = 500
SAVE_DIR = "model_final.pth"

# ---------------------------------------------------------------------------
# MUDANÇA 1/2/3: setup do process group + leitura de rank/local_rank/world_size
# ---------------------------------------------------------------------------
def ddp_setup():
    dist.init_process_group(backend="nccl")
    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    torch.cuda.set_device(local_rank)
    return rank, local_rank, world_size


def ddp_cleanup():
    dist.destroy_process_group()

# ---------------------------------------------------------------------------
# MUDANÇA 5: get_batch agora recebe rank/world_size e sorteia só dentro da
# fatia lógica do dataset que pertence a este processo.
# ---------------------------------------------------------------------------
data_dir = DATASET
def get_batch(split, batch_size, block_size, device, rank, world_size):
    filename = FILENAME if split == 'train' else 'val.bin'
    path = os.path.join(data_dir, filename)
    if split == 'val' and not os.path.exists(path):
        path = os.path.join(data_dir, FILENAME)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Arquivo de dados nao encontrado: {path}")

    data = np.memmap(path, dtype=np.uint16, mode='r')
    total_len = len(data) - block_size
    if total_len <= 0:
        raise ValueError(f"{path} possui poucos tokens para BLOCK_SIZE={block_size}.")

    # --- divisão lógica do dataset entre os ranks (sem shards físicos) ---
    shard_len = total_len // world_size
    shard_start = rank * shard_len
    shard_end = shard_start + shard_len
    # -----------------------------------------------------------------------

    ix = torch.randint(shard_start, shard_end, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])

    x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    return x, y


@torch.no_grad()
def evaluate(model, device, rank, world_size, num_batches=10):
    was_training = model.training
    model.eval()
    losses = []
    for _ in range(num_batches):
        x, y = get_batch("val", BATCH_SIZE, BLOCK_SIZE, device, rank, world_size)
        _, loss = model(x, y)
        losses.append(loss.item())
    if was_training:
        model.train()
    return sum(losses) / len(losses)

def main():
    # MUDANÇA 3/4: cada processo pega seu rank e sua própria GPU
    rank, local_rank, world_size = ddp_setup()
    device = torch.device(f"cuda:{local_rank}")
    is_main_process = (rank == 0)

    config = Config({
        "vocab_size": VOCAB_SIZE,
        "embedding_dim": EMBEDDING_DIM,
        "num_heads": NUM_HEADS,
        "num_layers": NUM_LAYERS,
        "block_size": BLOCK_SIZE,
        "dropout": DROPOUT,
    })

    model = ModeloCompleto(config).to(device)

    # MUDANÇA 6: DDP no lugar de DataParallel
    model = DDP(model, device_ids=[local_rank])

    if is_main_process:
        parametros = int(sum(p.numel() for p in model.parameters()))
        print(f"Modelo: DDP | Parâmetros: {parametros:,}")
        print(f"World size: {world_size} | Batch local: {BATCH_SIZE} | "
              f"Batch efetivo global: {BATCH_SIZE * world_size}")
        print(f"{MAX_STEPS} steps | Grad Accum Steps: {GRAD_ACCUM_STEPS} | "
              f"LR: {LEARNING_RATE} | Min LR: {MIN_LR} | Warmup Steps: {WARMUP_STEPS}")
        print()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        betas=(0.9, 0.95),
        eps=1e-8,
        weight_decay=float(WEIGHT_DECAY),
    )

    model.train()
    step = 0
    val_loss = 0.0
    # Cada optimizer.step processa estes tokens em TODAS as GPUs.  Este e o
    # valor usado tanto para tokens vistos como para throughput global.
    tokens_por_step = BATCH_SIZE * BLOCK_SIZE * GRAD_ACCUM_STEPS * world_size
    tokens_vistos = 0

    raw_model = model.module  # <-- para salvar, sempre acesse o modelo "puro" via .module

    while step < MAX_STEPS:
        # Operacoes CUDA sao assincronas. Sincronizar nas duas pontas faz dt
        # representar o tempo real do step, incluindo forward/backward/DDP.
        torch.cuda.synchronize(device)
        inicio_step = time.perf_counter()

        optimizer.zero_grad(set_to_none=True)
        step_loss_accum = 0.0

        lr = get_lr(step, LEARNING_RATE, WARMUP_STEPS, MAX_STEPS, MIN_LR)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        for micro_step in range(GRAD_ACCUM_STEPS):
            x, y = get_batch("train", BATCH_SIZE, BLOCK_SIZE, device, rank, world_size)

            logits, loss = model(x, y)

            if not torch.isfinite(loss):
                raise FloatingPointError(f"Loss não finita no step {step}: {loss.item()}")

            step_loss_accum += loss.item()

            loss_scaled = loss / GRAD_ACCUM_STEPS
            loss_scaled.backward()
            # MUDANÇA 6 (cont.): o all-reduce dos gradientes entre GPUs acontece
            # automaticamente aqui dentro do .backward() do DDP.

        if GRAD_CLIP is not None:
            # clip_grad_norm_ retorna a norma ANTES do clipping.
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        else:
            # Sem clipping, um limite infinito mede a mesma norma sem altera-la.
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), float("inf"))

        optimizer.step()
        torch.cuda.synchronize(device)
        dt = time.perf_counter() - inicio_step
        tokens_vistos += tokens_por_step
        tokens_por_segundo = tokens_por_step / dt if dt > 0 else float("inf")

        # MUDANÇA 8: eval, prints e save só no rank 0
        if is_main_process:
            if step % EVAL_INTERVAL == 0:
                val_loss = evaluate(model, device, rank, world_size)
                print(f"Validação | Step {step} | Val Loss: {val_loss:.4f} | LR: {lr:.10f}")

            if step % PRINT_INTERVAL == 0:
                print(f"Step {step} | Loss: {step_loss_accum / GRAD_ACCUM_STEPS:.4f} | "
                      f"VAL Loss: {val_loss:.4f} | Grad norm: {grad_norm.item():.4f} | "
                      f"dt: {dt * 1000:.2f} ms | tok/s: {tokens_por_segundo:,.0f} | "
                      f"tokens vistos: {tokens_vistos:,} | LR: {lr:.10f}")

        step += 1

    if is_main_process:
        print("Treinamento finalizado.")
        save_model(SAVE_DIR, raw_model)

    ddp_cleanup()


if __name__ == "__main__":
    main()
    # Para a execução
    exit()


In [ ]:
# Comando para rodar DDP em Kaggle, sem o comando "torchrun não é possivel fazer DDP"

!torchrun --nproc_per_node=2 /kaggle/working/.virtual_documents/__notebook_source__.ipynb
# torchrun --nproc_per_node=2 train_ddp.py